In [6]:
#1. the following improts are used for the first and most crucial astep, creating embeddings using a pprotein language model.
#Also for complete referece please note that the length of the protein is 305 AMINO ACIDS LONG (I have checked on pymol first)#
import torch 
from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM
from Bio.PDB import PDBParser, PPBuilder

In [3]:
#The protein itself is highly reliable since its refinement resolution is 0.92 Å. It is laos worthwehile to note that Glutamic acid at position 149 had become partailly decarboxylated under the X-ray diffraction.
def getting_wildtype_sequence(pdb_path):
    parser = PDBParser(QUIET = True)
    structure = parser.get_structure("2FVY", pdb_path)


    #I will now extract chain A from the amino acid seqence
    polypeptide_builder = PPBuilder()
    sequences = []

    residue_to_string = {}
    string_index = 0

    for pp in polypeptide_builder.build_peptides(structure[0]['A']): #Keep in mind there is only once chain and that is chain A as pary. Please referencfe pymol to confirm this fact. 
        sequences.append(str(pp.get_sequence()))

        #the following for loop takes out the exact PDB residue numebrs for alignment#
        for residue in pp:
            number_of_residues = residue.id[1]
            residue_to_string[number_of_residues] = string_index
            string_index += 1
    
    return "".join(sequences), residue_to_string

#Since I have now built the extrasction fucntion to get the full CHian A in the GGBP protein. I will call the fucntionm with the

protein = "2FVY.pdb"
wildtype_sequence, residue_map = getting_wildtype_sequence(protein)
print(f"Wild-Type Sequence (Length {len(wildtype_sequence)}):\n{wildtype_sequence}\n")
#The results confirm proper data rtransfer since pymol confrims as well it is 305 amino acids long. 

Wild-Type Sequence (Length 305):
DTRIGVTIYKYDDNFMSVVRKAIEQDAKAAPDVQLLMNDSQNDQSKQNDQIDVLLAKGVKALAINLVDPAAAGTVIEKARGQNVPVVFFNKEPSRKALDSYDKAYYVGTDSKESGIIQGDLIAKHWAANQGWDLNKDGQIQFVLLKGEPGHPDAEARTTYVIKELNDKGIKTEQLQLDTAMWDTAQAKDKMDAWLSGPNANKIEVVIANNDAMAMGAVEALKAHNKSSIPVFGVDALPEALALVKSGALAGTVLNDANNQAKATFDLAKNLADGKGAADGTNWKIDNKVVRVPYVGVDKDNLAEF



In [37]:
#The paper that has been referenced for past mutqations is the one linked on the RCSB databank file. You can find it here: (https://pmc.ncbi.nlm.nih.gov/articles/PMC2206672/) 
#The three residues identified in (Borrock et al 2007) that are of great importance (since they perform direct glucose contacts) are: Trp183 (Tryptophan at position 183), Phe16 (Phenylalanine at Poistion 16), and Asp154 (Aspartic Acid at positon 154)
# Hinge Sites worth being considered in comparison to 2FWO (the apo, unliganded version of the protein) are: between position 109–111, 253–256, and 293–296
#when I write aa in variables it is short-term for amino acid


# 1. Adjust mutations to create ensure safe glucose contact
def generate_mutant_sequence_safe(wildtype_sequence, residue_map, mutation_str):
    """
    mutation_str format: 'F16A', 'D154A', 'W183A'
    """
    expected_wildtype_amino_acid = mutation_str[0]
    pdb_residue_number = int(mutation_str[1:-1])
    mutant_amino_acid = mutation_str[-1]
    
    if pdb_residue_number not in residue_map:
        raise ValueError(f"Residue number {pdb_residue_number} not found in the PDB chain records.")
        
    string_position = residue_map[pdb_residue_number]
    actual_wildtype_amino_acid = wildtype_sequence[string_position]
    
    if actual_wildtype_amino_acid != expected_wildtype_amino_acid:
        print(f"Warning: Reference mismatch at site {pdb_residue_number}. Found {actual_wildtype_amino_acid}, expected {expected_wildtype_amino_acid}")
        
    mutant_sequence_list = list(wildtype_sequence)
    mutant_sequence_list[string_position] = mutant_amino_acid
    return "".join(mutant_sequence_list), string_position

mutations_to_run = ["F16A", "D154A", "W183A"]
sequences = {"Wild-Type": wildtype_sequence}
mutation_positions = {}

print("Step 1: Preparing mutant sequences for direct glucose contacts...")
for mutation in mutations_to_run:
    mutant_sequence, string_position = generate_mutant_sequence_safe(wildtype_sequence, residue_map, mutation)
    sequences[f"Mutant_{mutation}"] = mutant_sequence
    mutation_positions[mutation] = string_position
print("-> Mutant sequences successfully compiled.\n")


#ESM-2 embedding pipeline. In short what thgis does is basically convert a string of amino acids into vector quantities. With the embeddings the model begins to understand biology like hwat amino acids will be most likely to fold.
print("Step 2: Loading ESM-2 model from Hugging Face...")
model_name = "facebook/esm2_t12_35M_UR50D" #This calls ESM2 thorugh its public api key. 
tokenizer = AutoTokenizer.from_pretrained(model_name) #USing the UAto Tokenizer from the transformers library allows all the amino acid sequences to be converted into numbers immediately and will be sued against the ESM2 Hugging Face Repository model.
model = EsmModel.from_pretrained(model_name)

device = torch.device("cpu") #This ensures the device set is cpu. IF anyoine that has NVIDIA GPU would like to run please switch to GPU/
model = model.to(device)
model.eval()

print("\nStep 3: Extracting latent space representations...")
embeddings_registry = {}
for name, sequence_string in sequences.items():
    inputs = tokenizer(sequence_string, return_tensors="pt", add_special_tokens=True).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Slice [0, 1:-1, :] to remove special tokens so the array indices stay aligned 1:1 with wildtype_sequence#
    token_embeddings = outputs.last_hidden_state[0, 1:-1, :].cpu().numpy()
    embeddings_registry[name] = {"residue_embeddings": token_embeddings}
    print(f"Generated embedding matrix for: {name}")


# 3. TARGETED CLEFT BOUND METRIC ANALYSIS
print("\nStep 4: Analyzing target structural landmarks from 2FVY high-resolution data")

analysis_results = {}
for mutation in mutations_to_run:
    index = mutation_positions[mutation]
    wildtype_vector = embeddings_registry["Wild-Type"]["residue_embeddings"][index]
    mutant_vector = embeddings_registry[f"Mutant_{mutation}"]["residue_embeddings"][index]
    
    similarity_score = torch.nn.functional.cosine_similarity(
        torch.tensor(wildtype_vector), torch.tensor(mutant_vector), dim=0
    ).item()
    analysis_results[mutation] = similarity_score #This assigns the similarity score to the [mutation] array which contains all teh scalar values of result./.

print("2FVY ESM-2 Embedding Results: ")
for mutation, score in analysis_results.items():
    print(f"Mutation {mutation:<6} | Local Site Cosine Similarity: {score:.4f}")

Step 1: Preparing mutant sequences for direct glucose contacts...
-> Mutant sequences successfully compiled.

Step 2: Loading ESM-2 model from Hugging Face...


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Step 3: Extracting latent space representations...
Generated embedding matrix for: Wild-Type
Generated embedding matrix for: Mutant_F16A
Generated embedding matrix for: Mutant_D154A
Generated embedding matrix for: Mutant_W183A

Step 4: Analyzing target structural landmarks from 2FVY high-resolution data
2FVY ESM-2 Embedding Results: 
Mutation F16A   | Local Site Cosine Similarity: 0.7779
Mutation D154A  | Local Site Cosine Similarity: 0.8565
Mutation W183A  | Local Site Cosine Similarity: 0.7994


In [ ]:
#I will now create Masked Learning Model with logits (basically utilized log for predicition, and sees if it will increase). This is a continuation from teh last cell and will then allow me to print the mutations and test them using pymol and chimera.
# Also, cosine cimilarity is used on a tensor progbability and is used throughout transformer models required to run these ESM2 API keys. 
#Here mlm in a variable which stands for Masked Machine Learning. Masked machine learning uses a training technique which involves input data being intentionally hidden so that the model is forced to predict what happens after.

model_maskedmachinelearning = EsmForMaskedLM.from_pretrained(model_name) #The model_name is called from previous cll and represents the public API key for ESM2
model_maskedmachinelearning = model_maskedmachinelearning.to(device)
model_maskedmachinelearning.eval()

#I will now basicaly create a set standard towards all mutations after evaluating teh cosine similarity in the previous cell. This will help us create proper results.
target_panel = {
    "F16A": {"role": "Cleft, aromatic stacking", "effect": "Affinity decrease, desired direction!"}, #(Borrock et. al 2007), please reference this to find the necessary simplified info.
    "W183A": {"role": "Cleft, aromatic stacking", "effect":"Affinity decrease, desired direction!"}, #(Borrock et. al 2007), please reference this to find the necessary simplified info.
    "D154A": {"role": "Cleft, H-bond", "effect": "Minor effect"} #(Borrock et. al 2007), please reference this to find the necessary simplified info.
}

wildtypeinputs = tokenizer(wildtype_sequence, return_tensors="pt", add_special_tokens=True).to(device)
with torch.no_grad(): #This removes the gradient calculation for the model (the method of steepest descent). The gradient (in multivariable calculus) converts the partial derivative of the x,y,x with respect to eahc variable and converts it to a vector vlaue to determine the best way to reach the "minimum" of the fucntion. I like to think of it as the bottom of the bowl. What is crucial to know however is that when rtehre is no gradient indicates no backpropogation for inference#
    wildtype_outputs_mlm = model_maskedmachinelearning(**wildtypeinputs) #The gradient is not callculated here. Bascially, the evaluation of the ESM MLM (see previous cell for model evaluation). The training weight results from the previos are multiplied by the opriginal wild type sequence tensor. The traning weights are in matrices#
#The gradient not being clauclated is crucvisal sicne we extract the values later in the code and finding the gradient indicates we would be trying to find some form of error loss as well or tryign to evaluate the model. 

#I will now use logits (use log) to evaluate the most probable predicitions and mutations. It is used before any activation fucntion to ensure the unbiased, raw scores are set towards the neural netwrok.#
finaldatatable = []

for mutation in mutations_to_run:
    if mutation not in target_panel:
        continue 

    index = mutation_positions[mutation]

    #I will now erxtract the raw scores (logits) before applying the softmax activation function. 
    position_logits = wildtype_outputs_mlm.logits[0, index + 1, :]
    log_probability = torch.log_softmax(position_logits, dim = -1) #Dim represents the dimension of the fucntion. It is log inverse therefopre, e^x will be used.#

    wt_token_id = tokenizer.convert_tokens_to_ids(mutation[0])
    mut_token_id = tokenizer.convert_tokens_to_ids(mutation[-1])

    #I will now compute the zero shot socre using the following: Zero-shot score = ln(P(mutant)) - ln(P(wild-type))

    log_likelihood_ratio = (log_probability[mut_token_id] - log_probability[wt_token_id]).item() #Thsi is crucial sicne two tensors are subtracting and to pront the value we must convert it into a scalar value (in simple words a number) to allow pritning of data.
    cosine_similarity = analysis_results[mutation] #the cosine similarity results are pulled from the previous cell. #

    finaldatatable.append({
        "Mutation": mutation,
        "Site role": target_panel[mutation]["role"],
        "ESM2 log-likelihood": round(log_likelihood_ratio, 4),
        "Cosine similarity": round(cosine_similarity, 4),
        "Predicted effect": target_panel[mutation]["effect"]
    })

print("\nMasked Machine Learning Model on ESM-2 Embeddings Results (2FVY):")
print(f"{'Mutation':<10} | {'Site role':<25} | {'ESM2 log-likelihood':<20} | {'Cosine similarity':<17} | {'Predicted effect'}")
print("\n\n")
for entry in finaldatatable:
    print(f"{entry['Mutation']:<10} | {entry['Site role']:<25} | {entry['ESM2 log-likelihood']:<20} | {entry['Cosine similarity']:<17} | {entry['Predicted effect']}")

Loading weights:   0%|          | 0/214 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Masked Machine Learning Model on ESM-2 Embeddings Results (2FVY):
Mutation   | Site role                 | ESM2 log-likelihood  | Cosine similarity | Predicted effect



F16A       | Cleft, aromatic stacking  | -7.4465              | 0.7779            | Affinity decrease, desired direction!
D154A      | Cleft, H-bond             | -2.2361              | 0.8565            | Minor effect
W183A      | Cleft, aromatic stacking  | -6.6456              | 0.7994            | Affinity decrease, desired direction!


In [39]:
#I will now figure out what to change the amin io acids to optimize expression. It will be done by residue#
target_pdb_res = 16 
string_index = residue_map[target_pdb_res]

# Get the logits for that position
with torch.no_grad():
    # Offset by 1 for the <cls> token
    logits = wildtype_outputs_mlm.logits[0, string_index + 1, :]
    probs = torch.softmax(logits, dim=-1)

# Get top 5 most probable substitutions
top_k = 5
top_probs, top_indices = torch.topk(probs, top_k)

print(f"Top {top_k} amino acid recommendations for site {target_pdb_res}:")
for i in range(top_k):
    aa = tokenizer.decode(top_indices[i].item())
    score = top_probs[i].item()
    print(f"Amino Acid: {aa} | Probability: {score:.4f}")

Top 5 amino acid recommendations for site 16:
Amino Acid: F | Probability: 0.9341
Amino Acid: W | Probability: 0.0295
Amino Acid: Y | Probability: 0.0283
Amino Acid: L | Probability: 0.0015
Amino Acid: V | Probability: 0.0012


In [40]:
target_pdb_res = 154 
string_index = residue_map[target_pdb_res]

# Get the logits for that position
with torch.no_grad():
    # Offset by 1 for the <cls> token
    logits = wildtype_outputs_mlm.logits[0, string_index + 1, :]
    probs = torch.softmax(logits, dim=-1)

# Get top 5 most probable substitutions
top_k = 5
top_probs, top_indices = torch.topk(probs, top_k)

print(f"Top {top_k} amino acid recommendations for site {target_pdb_res}:")
for i in range(top_k):
    aa = tokenizer.decode(top_indices[i].item())
    score = top_probs[i].item()
    print(f"Amino Acid: {aa} | Probability: {score:.4f}")

Top 5 amino acid recommendations for site 154:
Amino Acid: D | Probability: 0.6669
Amino Acid: S | Probability: 0.0775
Amino Acid: A | Probability: 0.0713
Amino Acid: T | Probability: 0.0471
Amino Acid: V | Probability: 0.0266


In [41]:
target_pdb_res = 183
string_index = residue_map[target_pdb_res]

# Get the logits for that position
with torch.no_grad():
    # Offset by 1 for the <cls> token
    logits = wildtype_outputs_mlm.logits[0, string_index + 1, :]
    probs = torch.softmax(logits, dim=-1)

# Get top 5 most probable substitutions
top_k = 5
top_probs, top_indices = torch.topk(probs, top_k)

print(f"Top {top_k} amino acid recommendations for site {target_pdb_res}:")
for i in range(top_k):
    aa = tokenizer.decode(top_indices[i].item())
    score = top_probs[i].item()
    print(f"Amino Acid: {aa} | Probability: {score:.4f}")

Top 5 amino acid recommendations for site 183:
Amino Acid: W | Probability: 0.9379
Amino Acid: Y | Probability: 0.0242
Amino Acid: F | Probability: 0.0219
Amino Acid: N | Probability: 0.0037
Amino Acid: G | Probability: 0.0028
